# Experimento — KAN, fase 4: validação interna e confirmação

Esta fase troca a escolha por validação direta por um protocolo honesto: em cada dobra externa, 80% da porção de treino ajusta a KAN e 20% escolhe o número de passos pela BCE. A KAN é então retreinada na dobra externa inteira com esse número de passos. A dobra externa mede; não decide.

As divisões desta fase são novas (sementes 201–203), distintas do placar final. O teste oficial continua fechado.

## 0. Preparação e configuração congelada para a confirmação

In [ ]:
import copy
import json
import os
import sys
from pathlib import Path

CSV_PATH = os.environ.get("CHURN_CSV", "")
if CSV_PATH:
    os.environ["CHURN_CSV"] = CSV_PATH
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "comum.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Abra o notebook dentro do repositório churn_project.")
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import comum
from experimentos.kan import modelo as K

RESULTADOS = Path(comum.RESULTADOS)
PASTA_KAN = RESULTADOS / "kan"
PASTA_KAN.mkdir(parents=True, exist_ok=True)
print(f"Repositório: {ROOT}")
print(f"Dispositivo: {K.DEVICE}")
print(f"CSV: {comum.CAMINHO_CSV}")

In [ ]:
ARQUIVO = PASTA_KAN / "kan_fase3.json"
if not ARQUIVO.is_file():
    raise FileNotFoundError(f"Execute antes a fase anterior: {ARQUIVO}")
REGISTRO = json.loads(ARQUIVO.read_text(encoding="utf-8"))
CAMPEAO = dict(REGISTRO["config"])
CAMPEAO["hidden"] = tuple(CAMPEAO["hidden"])
FEATURES = tuple(REGISTRO.get("features", []))
OVER = REGISTRO.get("over", True)
print("Configuração recebida:", CAMPEAO)
print("Features:", FEATURES, "| oversampling:", OVER)

MODO_RAPIDO = False
SEMENTES_CONFIRM = (201,) if MODO_RAPIDO else (201, 202, 203)
X_train, _, y_train, _ = comum.dividir(comum.carregar_df())
divisoes = comum.divisoes_placar(y_train.to_numpy(), sementes=SEMENTES_CONFIRM)
print("Divisões de confirmação:", len(divisoes))

## 1. Confirmação sem usar as dobras para decidir os passos

In [ ]:
medida = K.avaliar_placar(CAMPEAO, FEATURES, OVER, divs=divisoes)
resumo = {k: v for k, v in medida.items() if k != "oof"}
display(pd.DataFrame({"KS por dobra": medida["ks_dobras"]}))
print("KS k-fold:", medida["ks_kfold"], "± erro padrão:", medida["ep"])
print("KS OOF:", medida["ks_oof"], "| AUROC OOF:", medida["auc_oof"])
print("Passos medianos escolhidos somente nas partições internas:", medida["steps_mediana"])

## 2. Verificação de estabilidade

In [ ]:
kss = np.asarray(medida["ks_dobras"], dtype=float)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.arange(1, len(kss) + 1), kss, "o-", label="KAN retreinada")
ax.axhline(kss.mean(), color="black", linestyle="--", label=f"média = {kss.mean():.4f}")
ax.set(title="Confirmação em divisões não usadas na busca", xlabel="divisão", ylabel="KS externo")
ax.legend()
ax.grid(alpha=.25)
plt.tight_layout()

## 3. Persistir configuração e protocolo

In [ ]:
saida = {"config": {**CAMPEAO, "hidden": list(CAMPEAO["hidden"])},
         "features": list(FEATURES), "over": OVER,
         "confirmacao": {k: v for k, v in resumo.items() if k != "steps_por_dobra"},
         "sementes": list(SEMENTES_CONFIRM), "modo_rapido": MODO_RAPIDO}
(PASTA_KAN / "kan_fase4.json").write_text(json.dumps(saida, ensure_ascii=False, indent=2), encoding="utf-8")
print("Salvo em:", PASTA_KAN / "kan_fase4.json")